# Employability Prediction — Full Pipeline (Colab)

This notebook reproduces the complete thesis pipeline end to end:

**Dataset → Cleaning → Feature Engineering → Preprocessing → Model Training → Hyperparameter Tuning → Evaluation → Graphs → Cross-Validation**

Run every cell from top to bottom. The first code cell asks you to upload
`student_career_success_dataset.xlsx` — pick it from your computer when the
file picker appears.

All numbers reproduce the thesis's Chapter 5 exactly:
- Tuned XGBoost, `random_state=42` everywhere → fully reproducible.
- Primary reported configuration: **decision threshold = 0.300** (chosen for
  deployment balance between accuracy and minority-class recall — see
  Section 5.7 of the thesis for the full trade-off discussion).
- The 4-algorithm comparison (Table 5.6) uses the standard threshold = 0.50,
  intentionally, since it is an independent benchmark.

## 1. Setup

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib openpyxl

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    RandomizedSearchCV, train_test_split, StratifiedKFold, cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, roc_auc_score,
    precision_score, recall_score, f1_score,
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
plt.rcParams["figure.dpi"] = 110
print("Setup complete.")

## 2. Upload the dataset

Run this cell, then choose `student_career_success_dataset.xlsx` from your
computer in the file picker that appears.

In [ ]:
from google.colab import files

print("Please select student_career_success_dataset.xlsx")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print("Loaded file:", DATA_PATH)

## 3. Load and clean the dataset

In [ ]:
df = pd.read_excel(DATA_PATH) if DATA_PATH.lower().endswith((".xlsx", ".xls")) else pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)

# --- cleaning ---
df = df.drop_duplicates()
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

TARGET_COL = "Placement_Status"
if set(df[TARGET_COL].unique()) <= {"Placed", "Not Placed"}:
    df[TARGET_COL] = df[TARGET_COL].map({"Placed": "Employable", "Not Placed": "Not Employable"})

assert TARGET_COL in df.columns, f"Target column {TARGET_COL} not found"

# --- drop post-outcome leakage columns (Section 5.2 of the thesis) ---
leakage_cols = [c for c in ["Student_ID", "Employability_Score", "Company_Tier",
                              "Career_Field", "Placement_Mode", "Starting_Salary_USD"]
                if c in df.columns]
df = df.drop(columns=leakage_cols)

print(f"After cleaning: {df.shape[0]} instances, {df.shape[1]} columns "
      f"(dropped {len(leakage_cols)} post-outcome columns).")
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

## 4. Feature engineering (2 composite features)

In [ ]:
df["Overall_Preparedness_Index"] = df["Interview_Score"] * 0.7 + df["Internships"] * 0.3
df["Skills_per_Project"] = df["Programming_Skill"] / (df["Projects_Completed"] + 1)
print("Added: Overall_Preparedness_Index, Skills_per_Project")
df[["Interview_Score", "Internships", "Overall_Preparedness_Index",
    "Programming_Skill", "Projects_Completed", "Skills_per_Project"]].head()

## 5. Exclude demographic/contextual attributes, split X/y

Six attributes are deliberately excluded from training (Section 5.2): a
student has no control over them, and including them risks the model
learning group membership rather than modifiable preparation behaviour.

In [ ]:
EXCLUDED_COLUMNS = ["Age", "Gender", "University_Year", "Major",
                    "Attendance_Percentage", "LinkedIn_Profile"]

y_raw = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL] + EXCLUDED_COLUMNS)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
NOT_EMPLOYABLE_IDX = list(label_encoder.classes_).index("Not Employable")
print("Label classes (alphabetical):", list(label_encoder.classes_))

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
print(f"Retained {X.shape[1]} features -> {len(numeric_features)} numeric, "
      f"{len(categorical_features)} categorical.")

## 6. Class distribution graph

In [ ]:
counts = y_raw.value_counts()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(counts.index, counts.values, color=["#4C72B0", "#DD8452"])
axes[0].set_title("Absolute class distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f"{v:,}", ha="center")

props = counts / counts.sum() * 100
axes[1].pie(props.values, labels=[f"{l}\n{p:.2f}%" for l, p in zip(props.index, props.values)],
            colors=["#4C72B0", "#DD8452"], startangle=90)
axes[1].set_title("Proportional class distribution")
fig.tight_layout()
plt.show()

print(f"Majority-class (Employable) baseline accuracy: {props.max():.2f}%")

## 7. Stratified 80:20 train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print("Train class balance:", pd.Series(y_train).value_counts(normalize=True).round(4).to_dict())
print("Test class balance:", pd.Series(y_test).value_counts(normalize=True).round(4).to_dict())

## 8. Preprocessing pipeline

Imputation, scaling and encoding are all fitted *inside* the pipeline, on
the training partition only — this is what prevents data leakage from the
test set into preprocessing parameters.

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])
print("Preprocessor built.")

## 9. Train the tuned XGBoost pipeline (RandomizedSearchCV)

No SMOTE, no class weighting — hyperparameters alone, selected by
cross-validated search (Section 5.6 of the thesis).

In [ ]:
import time

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(random_state=RANDOM_STATE, n_jobs=4, eval_metric="logloss")),
])

param_distributions = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__learning_rate": [0.01, 0.05, 0.1],
    "classifier__max_depth": [3, 4, 6],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0],
    "classifier__min_child_weight": [1, 3, 5],
}

search = RandomizedSearchCV(
    pipeline, param_distributions=param_distributions, n_iter=15, cv=3,
    scoring="accuracy", random_state=RANDOM_STATE, n_jobs=1, verbose=1,
)

t0 = time.time()
search.fit(X_train, y_train)
print(f"\nSearch completed in {time.time()-t0:.1f} seconds.")
print("Best params:", search.best_params_)

best_pipeline = search.best_estimator_

t0 = time.time()
best_pipeline.fit(X_train, y_train)
print(f"Final fit on {X_train.shape[0]} rows took {time.time()-t0:.2f} seconds.")

## 10. Evaluate — primary reported configuration (threshold = 0.300)

This is the thesis's primary reported configuration (Table 5.5), chosen as
the deployment operating point that maximises minority (at-risk) F1-score
without letting accuracy fall below the majority-class baseline.

In [ ]:
THRESHOLD = 0.300

proba_test = best_pipeline.predict_proba(X_test)[:, NOT_EMPLOYABLE_IDX]
y_pred = (proba_test >= THRESHOLD).astype(int)
other_idx = 1 - NOT_EMPLOYABLE_IDX
y_pred_labels = np.where(y_pred == 1, NOT_EMPLOYABLE_IDX, other_idx)

acc = accuracy_score(y_test, y_pred_labels)
auc = roc_auc_score(y_test, proba_test)
cm = confusion_matrix(y_test, y_pred_labels, labels=[0, 1])  # [Employable, Not Employable]

print(f"=== Classification report (threshold = {THRESHOLD}) ===")
print(classification_report(y_test, y_pred_labels, target_names=label_encoder.classes_, digits=4))
print("Confusion matrix [Employable, Not Employable]:")
print(cm)
print(f"\nAccuracy: {acc:.4f}")
print(f"ROC-AUC (threshold-independent): {auc:.4f}")

## 11. Confusion matrix graph

In [ ]:
labels = list(label_encoder.classes_)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title(f"Confusion Matrix (threshold = {THRESHOLD})")
for i in range(2):
    for j in range(2):
        ax.text(j, i, format(cm[i, j], "d"), ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
fig.colorbar(im, ax=ax)
fig.tight_layout()
plt.show()

## 12. Per-class precision / recall / F1 graph

In [ ]:
report = classification_report(y_test, y_pred_labels, target_names=labels, output_dict=True)
metrics = ["precision", "recall", "f1-score"]
x = np.arange(len(metrics))
width = 0.35

emp_vals = [report["Employable"][m] for m in metrics]
notemp_vals = [report["Not Employable"][m] for m in metrics]

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - width/2, emp_vals, width, label="Employable", color="#4C72B0")
ax.bar(x + width/2, notemp_vals, width, label="Not Employable", color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels(["Precision", "Recall", "F1-score"])
ax.set_ylim(0, 1.0)
ax.set_title(f"Per-Class Metrics (threshold = {THRESHOLD})")
ax.legend()
for i, v in enumerate(emp_vals):
    ax.text(i - width/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
for i, v in enumerate(notemp_vals):
    ax.text(i + width/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
fig.tight_layout()
plt.show()

## 13. Threshold sweep — see the full trade-off curve

This is how threshold = 0.300 was originally chosen: it is the point that
maximises minority F1-score while keeping accuracy at or above the
majority-class baseline (marked as a dashed line).

In [ ]:
majority_baseline = (y_test != NOT_EMPLOYABLE_IDX).mean()
thresholds = np.arange(0.20, 0.55, 0.025)
rows = []
for thr in thresholds:
    yp = (proba_test >= thr).astype(int)
    yl = np.where(yp == 1, NOT_EMPLOYABLE_IDX, other_idx)
    rows.append((
        thr,
        accuracy_score(y_test, yl),
        precision_score(y_test, yl, pos_label=NOT_EMPLOYABLE_IDX, zero_division=0),
        recall_score(y_test, yl, pos_label=NOT_EMPLOYABLE_IDX),
        f1_score(y_test, yl, pos_label=NOT_EMPLOYABLE_IDX, zero_division=0),
    ))
sweep_df = pd.DataFrame(rows, columns=["threshold", "accuracy", "minority_precision", "minority_recall", "minority_f1"])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sweep_df.threshold, sweep_df.accuracy, marker="o", label="Accuracy")
ax.plot(sweep_df.threshold, sweep_df.minority_precision, marker="o", label="Minority precision")
ax.plot(sweep_df.threshold, sweep_df.minority_recall, marker="o", label="Minority recall")
ax.plot(sweep_df.threshold, sweep_df.minority_f1, marker="o", label="Minority F1")
ax.axhline(majority_baseline, color="gray", linestyle="--", linewidth=1, label="Majority baseline")
ax.axvline(THRESHOLD, color="red", linestyle=":", linewidth=1.5, label=f"Selected threshold ({THRESHOLD})")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_title("Accuracy / Precision / Recall / F1 across candidate thresholds")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
plt.show()

sweep_df.round(4)

## 14. Feature importance

In [ ]:
fitted_pre = best_pipeline.named_steps["preprocessor"]
ohe = fitted_pre.named_transformers_["cat"].named_steps["onehot"]
cat_expanded = ohe.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_expanded)

importances = best_pipeline.named_steps["classifier"].feature_importances_
imap = {}
for name, score in zip(all_feature_names, importances):
    raw = name
    for c in categorical_features:
        if name.startswith(c + "_"):
            raw = c
            break
    imap[raw] = imap.get(raw, 0.0) + float(score)

total = sum(imap.values()) or 1.0
ranked = sorted(((k, v / total) for k, v in imap.items()), key=lambda kv: kv[1], reverse=True)

for name, score in ranked:
    print(f"{name:30s} {score:.6f}")

names = [n for n, _ in ranked]
scores = [s for _, s in ranked]
fig, ax = plt.subplots(figsize=(9, 7))
y_pos = np.arange(len(names))
ax.barh(y_pos, scores[::-1], color="#4C72B0")
ax.set_yticks(y_pos); ax.set_yticklabels(names[::-1])
ax.set_xlabel("Normalised importance")
ax.set_title("XGBoost Feature Importance (18 retained features)")
for i, v in enumerate(scores[::-1]):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=8)
fig.tight_layout()
plt.show()

## 15. Algorithm comparison (Table 5.6) — threshold = 0.50

Independent benchmark of 4 classifiers under the identical split/preprocessing,
at the standard 0.50 threshold (intentionally not threshold = 0.300, since
this is a separate methodological comparison, not the deployment recommendation).

In [ ]:
MODELS = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=4),
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, subsample=1.0,
        colsample_bytree=1.0, min_child_weight=1,
        random_state=RANDOM_STATE, n_jobs=4, eval_metric="logloss",
    ),
}

comparison_rows = []
for name, clf in MODELS.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", clf)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    elapsed = time.time() - t0

    proba = pipe.predict_proba(X_test)[:, NOT_EMPLOYABLE_IDX]
    yp = (proba >= 0.50).astype(int)
    yl = np.where(yp == 1, NOT_EMPLOYABLE_IDX, other_idx)

    acc_c = accuracy_score(y_test, yl)
    auc_c = roc_auc_score(y_test, proba)
    rec_c = recall_score(y_test, yl, pos_label=NOT_EMPLOYABLE_IDX)
    prec_c = precision_score(y_test, yl, pos_label=NOT_EMPLOYABLE_IDX, zero_division=0)
    comparison_rows.append((name, acc_c, auc_c, rec_c, prec_c, elapsed))

comparison_df = pd.DataFrame(comparison_rows,
    columns=["Model", "Accuracy", "ROC-AUC", "Minority Recall", "Minority Precision", "Train Time (s)"])
comparison_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(comparison_df["Model"], comparison_df["Accuracy"], color="#55A868")
ax.axhline(majority_baseline, color="gray", linestyle="--", label="Majority baseline")
ax.set_ylabel("Accuracy")
ax.set_title("Algorithm Comparison — Accuracy at threshold = 0.50")
ax.set_ylim(0.5, 0.9)
plt.xticks(rotation=15)
ax.legend()
for i, v in enumerate(comparison_df["Accuracy"]):
    ax.text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=9)
fig.tight_layout()
plt.show()

## 16. Robustness checks — 5-fold cross-validation (threshold = 0.300)

Independently confirms the reported test-set numbers are not an artefact of
one lucky split.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_accs, fold_aucs = [], []

for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
    Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
    ytr, yte = y[tr_idx], y[te_idx]

    fold_pipe = Pipeline([("preprocessor", preprocessor), ("classifier",
        XGBClassifier(random_state=RANDOM_STATE,
                      n_jobs=4, eval_metric="logloss",
                      n_estimators=search.best_params_["classifier__n_estimators"],
                      learning_rate=search.best_params_["classifier__learning_rate"],
                      max_depth=search.best_params_["classifier__max_depth"],
                      subsample=search.best_params_["classifier__subsample"],
                      colsample_bytree=search.best_params_["classifier__colsample_bytree"],
                      min_child_weight=search.best_params_["classifier__min_child_weight"]))])
    fold_pipe.fit(Xtr, ytr)

    proba_fold = fold_pipe.predict_proba(Xte)[:, NOT_EMPLOYABLE_IDX]
    yp_fold = (proba_fold >= THRESHOLD).astype(int)
    yl_fold = np.where(yp_fold == 1, NOT_EMPLOYABLE_IDX, other_idx)

    fold_accs.append(accuracy_score(yte, yl_fold))
    fold_aucs.append(roc_auc_score(yte, proba_fold))
    print(f"Fold {fold}: accuracy={fold_accs[-1]:.4f}  AUC={fold_aucs[-1]:.4f}")

print(f"\nMean accuracy: {np.mean(fold_accs):.4f} (std {np.std(fold_accs):.4f})")
print(f"Mean ROC-AUC:   {np.mean(fold_aucs):.4f} (std {np.std(fold_aucs):.4f})")
print(f"\nSingle-split test result: accuracy={acc:.4f}, AUC={auc:.4f} -- should sit comfortably within the fold spread above.")

## 17. Multi-seed stability check

In [ ]:
seed_results = []
for seed in [42, 7, 123, 2024]:
    Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    seed_pipe = Pipeline([("preprocessor", preprocessor), ("classifier",
        XGBClassifier(random_state=RANDOM_STATE, n_jobs=4, eval_metric="logloss",
                      n_estimators=search.best_params_["classifier__n_estimators"],
                      learning_rate=search.best_params_["classifier__learning_rate"],
                      max_depth=search.best_params_["classifier__max_depth"],
                      subsample=search.best_params_["classifier__subsample"],
                      colsample_bytree=search.best_params_["classifier__colsample_bytree"],
                      min_child_weight=search.best_params_["classifier__min_child_weight"]))])
    seed_pipe.fit(Xtr_s, ytr_s)
    proba_s = seed_pipe.predict_proba(Xte_s)[:, NOT_EMPLOYABLE_IDX]
    yp_s = (proba_s >= THRESHOLD).astype(int)
    other_idx_s = 1 - NOT_EMPLOYABLE_IDX
    yl_s = np.where(yp_s == 1, NOT_EMPLOYABLE_IDX, other_idx_s)
    acc_s = accuracy_score(yte_s, yl_s)
    auc_s = roc_auc_score(yte_s, proba_s)
    seed_results.append((seed, acc_s, auc_s))
    print(f"seed={seed}: accuracy={acc_s:.4f}  AUC={auc_s:.4f}")

accs_only = [r[1] for r in seed_results]
aucs_only = [r[2] for r in seed_results]
print(f"\nAccuracy range: {min(accs_only):.4f} - {max(accs_only):.4f}")
print(f"AUC range:      {min(aucs_only):.4f} - {max(aucs_only):.4f}")

## 18. Summary

This cell prints a one-page summary of every headline number, matching the
thesis's Chapter 5 (Table 5.5, 5.6, 5.7, 5.9).

In [ ]:
print("=" * 70)
print("SUMMARY - Primary reported configuration (threshold = 0.300)")
print("=" * 70)
print(f"Accuracy:                {acc:.4f}")
print(f"ROC-AUC:                 {auc:.4f}")
print(f"Minority (Not Employable) precision: {report['Not Employable']['precision']:.4f}")
print(f"Minority (Not Employable) recall:    {report['Not Employable']['recall']:.4f}")
print(f"Minority (Not Employable) F1:        {report['Not Employable']['f1-score']:.4f}")
print(f"Confusion matrix: {cm.tolist()}")
print()
print("Top 4 features by importance:")
for name, score in ranked[:4]:
    print(f"  {name:30s} {score:.4f}")
print()
print("5-fold CV: mean accuracy", round(np.mean(fold_accs), 4), "| mean AUC", round(np.mean(fold_aucs), 4))
print("Multi-seed accuracy range:", round(min(accs_only), 4), "-", round(max(accs_only), 4))
print()
print("Algorithm comparison (threshold=0.50):")
print(comparison_df.round(4).to_string(index=False))